# Serotonin MOD-1 and PDF-1/PDFR-1 in the full Julia worm

**Equations**: \(I_{MOD1,i}=g_i A_{5HT}(t)(v_i-E_{Cl})\); \(	au_c\dot c_i=lpha R_{PDF}(t)-c_i\); \(I_{PDF,i}=g_{PDF}c_i\) as an explicitly hypothetical bias. The simulated slow PDF state is evaluated analytically for piecewise-constant externally prescribed concentrations. **The default PDF neural gain is zero**; use nonzero gain only for an exploratory hypothesis. All runs use the original 279-neuron/24-segment model.

In [ ]:
from pathlib import Path
import os, time
import numpy as np
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'modWorm'/'__init__.py').exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within your modWorm checkout")
os.chdir(ROOT)

from modWorm import predefined_classes_nv as nv, utils
from modWorm import predefined_classes_mb as mb
from modWorm import proprioception_simulation as sim
from modWorm import animation
from modWorm.julia_multiscale import run_modulated_julia, run_independent_julia_population, JuliaRunSettings
from modWorm.serotonin.serotonin_modulation import SerotoninModulation
from modWorm.pdf_modulation import PDFR1Modulation

GAP, SYN = utils.construct_connectome_Cook()
MUSCLE_MAP = np.load('modWorm/muscle_maps/muscle_map_adjust.npy')
PRESET = np.load('modWorm/presets_input/input_mat_gentle_post_touch.npy')
print('Loaded canonical', GAP.shape, SYN.shape, MUSCLE_MAP.shape, PRESET.shape)

def make_worm(seed=17, *, x=0.0, y=0.0, angle=0.0):
    # Both model constructors consume NumPy global RNG (legacy contract).
    np.random.seed(seed)
    neural = nv.CelegansWorm_NervousSystem_PPC_Julia(GAP, SYN)
    body = mb.CelegansWorm_MuscleBody_PPC_Julia(MUSCLE_MAP)
    body.initcond = np.asarray(body.initcond, dtype=float).copy()
    body.initcond[0] += x
    body.initcond[1] += y
    body.initcond[2:2+body.body_SegCount] += angle
    return neural, body

# Each sample represents 0.01 s in the original PPC class.
STEPS = 4  # 0.03 simulated seconds. Increase only after profiling.
INPUTS = PRESET[:STEPS].copy()
SETTINGS = JuliaRunSettings()


In [ ]:
RUN_ALL = False  # Set True only after single baseline successfully completes.
CONDITIONS = {'baseline': (None,None),
 'serotonin': (lambda:SerotoninModulation(concentration=1.0),None),
 'pdf_signal_only': (None,lambda:PDFR1Modulation(concentration=1.0)),
 'combined': (lambda:SerotoninModulation(concentration=1.0),
              lambda:PDFR1Modulation(concentration=1.0,coupling_gain_pA_per_signal=0.1))}
results={}
for label,(s_factory,p_factory) in CONDITIONS.items():
    if not RUN_ALL and label != 'baseline':
        continue
    neuron,body=make_worm(seed=3)
    s=s_factory() if s_factory else None
    p=p_factory() if p_factory else None
    print('Running:',label)
    results[label]=run_modulated_julia(neuron,body,INPUTS,serotonin=s,pdfr1=p,settings=SETTINGS)
    print('  elapsed s:',results[label]['elapsed_seconds'])


In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
for label,r in results.items():
    ax.plot(r['x_solution'][:,0],r['y_solution'][:,0],label=label)
ax.set(xlabel='x (model units)',ylabel='y (model units)',title='Actual simulated head trajectories')
ax.legend();plt.show()

In [ ]:
if 'combined' in results:
    result=results['combined']
    fig,ax=plt.subplots(figsize=(7,3))
    ax.plot(result['time'],result['pdf_signal_solution'][:,0])
    ax.set(xlabel='Simulation time (s)', ylabel='Dimensionless PDF signal',title='Analytic first-order signal')
    plt.show()
# Optional: reuse original modWorm video function, but only with sufficient frames.
MAKE_VIDEO = False
if MAKE_VIDEO:
    from modWorm import muscle_body_params as params
    r=results['baseline']
    animation.animate_body(x=r['x_solution'],y=r['y_solution'],filename='julia_modworm_baseline',
                           xmin=-70,xmax=20,ymin=-20,ymax=70,
                           figsize_x=10,figsize_y=10,background_img_path=False,
                           animation_config=params.CE_animation)

**Interpretation caution:** The original 279-neuron model omits NSM, PDF targets/gain are exploratory, and the fluid-body equations remain the original reduced model. These runs do not validate roaming/dwelling or collective swarming.